# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [39]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

# Updated the vendor_id to "V-07" for the update

**One row of my result is one:** line item.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [40]:
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
import pandas as pd

orders_flat = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'] )

assert len(orders_flat) ==4 # expected 4 rows
assert orders_flat['qty'].sum()== 7 #quantities should sum to 7

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [41]:
# TODO
# Merge the vendor information on the vendor id (left join) -  turning the vendor info from json to data frame
joined = orders_flat.merge(pd.DataFrame(vendors_json), on='vendor_id', how='left', indicator=True )

# Determining which vendors did not match
joined[joined['_merge'] == 'left_only']

## Vendor Matches
# V-07 did not match (it is not in the vendor list). There are three units attached to it - Hot Dog, order 3.

,item,qty,order,vendor_id,name,zone,_merge
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [42]:
# TODO
# Keeping the unmatched row instead of dropping it - including the label with its vendor_id
# Why: NaN does not include which vendor is missing from the vendor, list so labeling helps to specify, and keeping the row allows the table to sum to 7 units.
joined['vendor_name'] = joined['name'].fillna('Unmatched: ' + joined['vendor_id'])

# Using the groupby function for the name and then summing over the quantity
quantity_per_vendor = joined.groupby('vendor_name')['qty'].sum()
print(quantity_per_vendor)

## Unmatched Vendor
# Cav Merch (1) & Hoos Burgers (3) match the vendor list; however, V-07 is not on the list, so its 3 units (hot dogs) show up as "Unmatched: V-07". Total = 7.

vendor_name
Cav Merch          1
Hoos Burgers       3
Unmatched: V-07    3
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [43]:
# TODO
# Using the groupby function for the zone and then summing over the quantity
zone_totals = joined.groupby('zone', dropna=False)['qty'].sum()
print(zone_totals)

## Zone Values
# Zone A produced the most sold items - 4 items in total. The zone total adds up to 4, not 7, which was previously calculated in question 1. The other three units come from V-07, which has no zone, since V-07 is not in the vendor list.

zone
A      4
NaN    3
Name: qty, dtype: int64


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [44]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
# Using a for loop to fix the missing lines key so I can use the pd.json_normalize() can work without crashing - {} keeps one row where item & qty are missing
for order in ragged_json:
    if 'lines' not in order:
        order['lines'] = [{}]

# Flattening the ragged_json with pd.json_normalize function
ragged_flat = pd.json_normalize(ragged_json, record_path='lines', meta=['order', 'vendor_id'] )
print(ragged_flat)

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1    NaN  NaN     5      V-10
2  Fries  NaN     6      V-01


### Q6 — Validate

In [45]:
assert len(orders_flat) == 4
assert orders_flat['qty'].sum() == 7
assert len(joined) == len(orders_flat), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

Question A) The "one row per line item" format helps analyze individual items and calculate quantities by vendor or zone. If we didn't use this approach and an order contained multiple items, it would be harder to determine how many of each item were sold.

Question B) A missing quantity is a unknown or not provided; however, a quantity of zero means that there were zero items sold. If we conflate them, such as treating missing quantities as zero, we could underestimate the total number of items sold, which would ultimately produce incorrect totals.